In [4]:
import pandas as pd

cohort9 = pd.read_excel('data/cohort9_cleaned.xlsx')
cohort10 = pd.read_excel('data/cohort10_cleaned.xlsx')

combined = pd.concat([cohort9, cohort10], ignore_index=True)

combined['Referral Source'] = combined['Referral Source'].replace({'Whatsapp Community': 'Whatsapp'})

channels = ['X (Twitter)', 'Whatsapp', 'Linkedin', 'Referral', 'Facebook', 'Instagram']

for channel in channels:
    combined[channel] = combined['Referral Source'] == channel  

channel_summary = combined.groupby(['Name', 'Email Address', 'Country'])[channels].any().reset_index()

first_registration_details = combined.sort_values('Timestamp').drop_duplicates(subset=['Name', 'Email Address', 'Country'], keep='first')

final_table = first_registration_details.merge(
    channel_summary, 
    on=['Name', 'Email Address', 'Country'], 
    suffixes=('_old', '')
)

final_table = final_table.drop(columns=[c + '_old' for c in channels])
final_table['Occupation'] = final_table['Occupation'].fillna('Not Collected')
final_table = final_table.drop(columns=['Amount Paid'])

final_table['channels_used'] = final_table[channels].sum(axis=1)

final_table['Day Time'] = final_table['Day Time'].replace({
    'Afternoon (12pm-6pm)': 'Afternoon',
    'Morning (5am-11am)': 'Morning',
    'Night (7pm-11pm)': 'Night',
    'Midnight (12am-4am)': 'Midnight'
})


final_table.to_excel('data/final_analysis_ready.xlsx', index=False)

print(final_table['Day Time'].unique())
print(final_table[channels].sum(axis=1).eq(0).sum())
print(final_table['Day Time'].isna().sum())

<StringArray>
['Morning', 'Night', 'Afternoon', 'Midnight']
Length: 4, dtype: str
0
0
